# Multimodal Meeting Intelligence Pipeline — run in Colab

This notebook clones and runs the pipeline end-to-end: **ingest → speech (ASR + diarization) → vision (scene detect + OCR) → LLM intelligence → evaluation**, on a free Colab GPU.

Repo: **[https://github.com/CarmiShimon/conversational_intelligence](https://github.com/CarmiShimon/conversational_intelligence)**

**Before running:** `Runtime` → `Change runtime type` → set **Hardware accelerator** to **GPU** (a T4 is fine — more headroom than the 4&nbsp;GB laptop GPU this project was designed around).

**You will need:**
- An **OpenAI API key** (for the LLM summary/topics/action-items stage).
- A **Hugging Face token**, with the model terms accepted for:
  [pyannote/speaker-diarization-3.1](https://hf.co/pyannote/speaker-diarization-3.1) and
  [pyannote/segmentation-3.0](https://hf.co/pyannote/segmentation-3.0) (click "Agree" on both — required or diarization silently falls back to a single speaker).

First-time setup (installing PyTorch + WhisperX + EasyOCR, etc.) takes ~5–10 minutes. A full run on the ~26&nbsp;minute sample clip takes ~5–10 minutes more on a T4.

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Clone the repository

In [ ]:
import os

REPO_URL = "https://github.com/CarmiShimon/conversational_intelligence.git"
REPO_DIR = "conversational_intelligence"

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL}
%cd {REPO_DIR}
!git pull --ff-only

## 3. System dependency: ffmpeg

In [ ]:
!apt-get -qq update && apt-get -qq install -y ffmpeg
!ffmpeg -version | head -n 1
!ffprobe -version | head -n 1

## 4. Python dependencies

PyTorch is installed **first**, pinned to the CUDA build in `constraints.txt` — WhisperX and friends must be installed against this exact torch, not whatever Colab ships by default (see `README.md` for why).

In [ ]:
!pip install -q torch==2.3.1 torchvision==0.18.1 torchaudio==2.3.1 \
    --index-url https://download.pytorch.org/whl/cu121
!pip install -q -r requirements.txt -c constraints.txt
!pip install -q -e . -c constraints.txt

## 5. Secrets

Preferred: add `OPENAI_API_KEY` and `HUGGINGFACE_TOKEN` via Colab's secret manager (key icon in the left sidebar) and grant this notebook access. If they're not there, you'll be prompted to paste them (hidden input, not saved in the notebook).

In [ ]:
import os
from pathlib import Path
from getpass import getpass


def get_secret(name: str, prompt: str) -> str:
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass
    return getpass(prompt)


openai_key = get_secret("OPENAI_API_KEY", "OpenAI API key (sk-...): ")
hf_token = get_secret("HUGGINGFACE_TOKEN", "Hugging Face token (hf_...): ")

os.environ["OPENAI_API_KEY"] = openai_key
os.environ["HUGGINGFACE_TOKEN"] = hf_token
Path(".env").write_text(
    f"OPENAI_API_KEY={openai_key}\nHUGGINGFACE_TOKEN={hf_token}\n", encoding="utf-8"
)
print("Secrets configured (not printed).")

## 6. Provide the input video

Defaults to the assignment's sample clip (auto-downloaded via `gdown`) so the notebook runs end-to-end with no upload needed. To use your own video instead, clear **drive_link** and re-run this cell — you'll be prompted to upload a file.

In [ ]:
drive_link = "https://drive.google.com/file/d/1OrMgA5-RXQKL4Db728WAMQXhYb-Dzdw7/view"  # @param {type:"string"}
language = "en"  # @param {type:"string"}

from pathlib import Path

if drive_link.strip():
    video_source = drive_link.strip()
else:
    from google.colab import files
    print("Upload a video file (mp4/mov/etc.):")
    uploaded = files.upload()
    fname = next(iter(uploaded))
    Path("data/input").mkdir(parents=True, exist_ok=True)
    dest = Path("data/input") / fname
    dest.write_bytes(uploaded[fname])
    video_source = str(dest)

print("Using video source:", video_source)

## 7. Run the pipeline

Stage artifacts are cached under `outputs/cache/`, so re-running this cell after a partial failure (e.g. an LLM quota error) skips the already-completed stages instead of redoing the expensive ASR/diarization pass. Pass `--force` to ignore the cache.

In [ ]:
lang_flag = f"--language {language}" if language.strip() else ""
!python -m mmi.run --input "{video_source}" --output outputs {lang_flag}

## 8. Inspect the results

In [ ]:
import json

result = json.load(open("outputs/result.json", encoding="utf-8"))
meta, transcript, visual, intel = (
    result["metadata"], result["transcript"], result["visual"], result.get("intelligence")
)

print(f"Language: {transcript['language']}  |  Duration: {meta.get('duration_sec', 0) / 60:.1f} min")
print(f"Speakers: {sorted(set(s['speaker'] for s in transcript['segments']))}")
print(f"Scenes: {len(visual['scenes'])}  |  Stage seconds: {meta.get('stage_seconds')}")
print()

if intel:
    print("=== SUMMARY ===")
    print(intel["summary"])
    print("\n=== TOPICS ===")
    for t in intel["topics"]:
        print(" -", t["text"])
    print("\n=== ACTION ITEMS ===")
    for a in intel["action_items"]:
        owner = f" (owner: {a['owner']})" if a.get("owner") else ""
        print(" -", a["text"] + owner)
    print("\n=== DECISIONS ===")
    for d in intel["decisions"]:
        print(" -", d["text"])
else:
    print("LLM stage did not produce output. metadata.llm_error:", meta.get("llm_error"))

Optional: preview a representative keyframe.

In [ ]:
from IPython.display import Image, display

scenes_with_kf = [s for s in visual["scenes"] if s.get("keyframe_path")]
if scenes_with_kf:
    display(Image(filename=scenes_with_kf[len(scenes_with_kf) // 2]["keyframe_path"]))
else:
    print("No keyframes were extracted.")

## 9. (Optional) Evaluate against a reference

Only meaningful if you ran the **default sample clip** above — `data/reference/reference.json` is a bootstrap reference specific to that recording (see `docs/report.md` for why WER/speaker-accuracy against it are harness-validation, not accuracy claims). Skip this cell if you uploaded your own video.

In [ ]:
!python -m mmi.evaluate --result outputs/result.json --reference data/reference/reference.json --out outputs/eval.json

## 10. Download the outputs

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("mmi_outputs", "zip", "outputs")
files.download("mmi_outputs.zip")